# SQL for Data Platforms
## Part 8: Slowly Changing Dimensions & Incremental/Delta Loading

This is the module that ties together a question that trips a lot of people up in interviews:
*"how do you handle a dimension that changes over time, inside an incremental ETL/ELT pipeline?"*
Two ideas that are almost always asked about together, but taught separately: **Slowly Changing
Dimensions** (a schema-design question — Part 2) and **incremental/delta loading** (a pipeline
question — see [ETL vs. ELT](../../mfzamudio.github.io/publications/pattern-etl-vs-elt.html) and
[Orchestration & CDC](../../mfzamudio.github.io/publications/pattern-orchestration-cdc.html) on
the main site for the platform-architecture framing). This module puts them in one place, with
real SQL.

## Setup — reusing Part 2's star schema

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')

def sql(query):
    return pd.read_sql_query(query, conn)

def execute(statement):
    conn.execute(statement)
    conn.commit()

execute("""
CREATE TABLE dim_customer (
    customer_key     INTEGER PRIMARY KEY,
    customer_id      INTEGER NOT NULL,
    name             VARCHAR(100) NOT NULL,
    city             VARCHAR(50),
    membership_level VARCHAR(10)
)
""")
execute("""
INSERT INTO dim_customer VALUES
    (1, 1, 'Alice Martin', 'Toronto', 'premium'),
    (2, 7, 'Grace Kim', 'Vancouver', 'vip')
""")
print("dim_customer ready (from Part 2)")
sql("SELECT * FROM dim_customer")

dim_customer ready (from Part 2)


,customer_key,customer_id,name,city,membership_level
0,1,1,Alice Martin,Toronto,premium
1,2,7,Grace Kim,Vancouver,vip


## Section 1 — Full load vs. incremental load

**Full load:** truncate the target and reload every row from the source, every run. Simple, always
correct, but scales badly — reloading a 500M-row table nightly to pick up 5,000 changed rows wastes
time and compute.

**Incremental load:** load only the rows that *changed* since the last run. Requires a way to
detect what changed:

| Technique | How it works | Tradeoff |
|---|---|---|
| **Watermark** | Filter source on `updated_at > last_load_timestamp` | Needs a trustworthy, always-updated timestamp column; misses hard deletes |
| **Hash-diff** | Hash every row's columns; compare to last run's hash to spot changes | Catches changes without a reliable timestamp; more compute per run |
| **Log-based CDC** | Read the source database's transaction log directly | Catches inserts/updates/*deletes*; needs CDC infrastructure (Debezium, native CDC) — see [Orchestration & CDC](../../mfzamudio.github.io/publications/pattern-orchestration-cdc.html) |

Incremental loading is a **mechanism**. Slowly Changing Dimensions is the **policy** for what to do
with a changed dimension row once you've detected it.

In [2]:
# Watermark-style delta detection — simulate a source table with an updated_at column
execute("""
CREATE TABLE source_customers (
    customer_id INTEGER PRIMARY KEY,
    name        VARCHAR(100),
    city        VARCHAR(50),
    membership_level VARCHAR(10),
    updated_at  TIMESTAMP
)
""")
execute("""
INSERT INTO source_customers VALUES
    (1, 'Alice Martin', 'Toronto',  'premium', '2024-01-10 09:00:00'),
    (7, 'Grace Kim',    'Montreal', 'vip',     '2024-06-15 14:30:00')  -- Grace MOVED cities
""")

last_load_timestamp = '2024-03-01 00:00:00'  # watermark from the previous run
sql(f"""
SELECT * FROM source_customers WHERE updated_at > '{last_load_timestamp}'
""")

,customer_id,name,city,membership_level,updated_at
0,7,Grace Kim,Montreal,vip,2024-06-15 14:30:00


Only Grace's row comes back — Alice hasn't changed since the last load, so a watermark-based
pipeline skips her entirely. That's the whole point: **1 row processed, not 2.**

## Section 2 — SCD Type 1: overwrite (no history)

Simplest policy: when a dimension attribute changes, just `UPDATE` it in place. The old value is
gone. Use this for corrections (a typo fix) or attributes where history genuinely doesn't matter.

In [3]:
execute("""
UPDATE dim_customer
SET city = (SELECT city FROM source_customers WHERE source_customers.customer_id = dim_customer.customer_id)
WHERE customer_id = 7
""")
sql("SELECT * FROM dim_customer WHERE customer_id = 7")
# Montreal now — Vancouver is gone forever. Fine for a typo fix; wrong if you ever need to ask
# "what region was this customer in when they placed order #17?" 

,customer_key,customer_id,name,city,membership_level
0,2,7,Grace Kim,Montreal,vip


## Section 3 — SCD Type 2: full version history (the one interviews ask about)

Keep **every version** of a row, each with its own surrogate key and a validity window. This is
the pattern that answers "what was true *at the time*" — essential for anything you'll ever
re-analyze historically (which region a sale should be attributed to, what tier a customer was in
when they churned, etc.).

**Schema additions:** `effective_date`, `end_date`, `is_current` (or `valid_from`/`valid_to`).
`end_date IS NULL` (or a far-future sentinel) marks the current row.

In [4]:
execute("""
CREATE TABLE dim_customer_scd2 (
    customer_key     INTEGER PRIMARY KEY,
    customer_id      INTEGER NOT NULL,
    name             VARCHAR(100) NOT NULL,
    city             VARCHAR(50),
    membership_level VARCHAR(10),
    effective_date   DATE NOT NULL,
    end_date         DATE,               -- NULL means "still current"
    is_current       INTEGER NOT NULL DEFAULT 1
)
""")
execute("""
INSERT INTO dim_customer_scd2 VALUES
    (1, 7, 'Grace Kim', 'Vancouver', 'vip', '2023-02-20', NULL, 1)
""")
sql("SELECT * FROM dim_customer_scd2")

,customer_key,customer_id,name,city,membership_level,effective_date,end_date,is_current
0,1,7,Grace Kim,Vancouver,vip,2023-02-20,None,1


### The Type 2 upsert

The modern, one-statement way to express this is a **`MERGE`** (ANSI SQL:2003 — supported by
PostgreSQL 15+, SQL Server, Oracle, Snowflake, BigQuery; **not** SQLite):

```sql
MERGE INTO dim_customer_scd2 AS target
USING source_customers AS src
ON target.customer_id = src.customer_id AND target.is_current = 1
WHEN MATCHED AND target.city != src.city THEN
    UPDATE SET end_date = CURRENT_DATE, is_current = 0
WHEN NOT MATCHED THEN
    INSERT (customer_id, name, city, membership_level, effective_date, is_current)
    VALUES (src.customer_id, src.name, src.city, src.membership_level, CURRENT_DATE, 1);
-- a second INSERT ... SELECT is still needed for the new current row in most MERGE dialects,
-- since one MERGE statement can only take one action per matched row
```

SQLite has no `MERGE`, so here is the equivalent, explicit two-statement transaction — **this is
exactly what a `MERGE` compiles down to under the hood on the platforms that have it**:

In [5]:
new_city = sql("SELECT city FROM source_customers WHERE customer_id = 7").iloc[0, 0]
current = sql("SELECT * FROM dim_customer_scd2 WHERE customer_id = 7 AND is_current = 1")
changed = current.iloc[0]['city'] != new_city
print(f"current city on file: {current.iloc[0]['city']!r}  |  source city: {new_city!r}  |  changed: {changed}")

if changed:
    execute("""
        UPDATE dim_customer_scd2
        SET end_date = '2024-06-15', is_current = 0
        WHERE customer_id = 7 AND is_current = 1
    """)
    execute(f"""
        INSERT INTO dim_customer_scd2
            (customer_id, name, city, membership_level, effective_date, end_date, is_current)
        VALUES (7, 'Grace Kim', '{new_city}', 'vip', '2024-06-15', NULL, 1)
    """)

sql("SELECT * FROM dim_customer_scd2 WHERE customer_id = 7 ORDER BY effective_date")

current city on file: 'Vancouver'  |  source city: 'Montreal'  |  changed: True


,customer_key,customer_id,name,city,membership_level,effective_date,end_date,is_current
0,1,7,Grace Kim,Vancouver,vip,2023-02-20,2024-06-15,0
1,2,7,Grace Kim,Montreal,vip,2024-06-15,NaN,1


In [6]:
# The payoff: "what city was Grace in on 2024-03-01?" is now answerable, historically
sql("""
SELECT city, effective_date, end_date
FROM dim_customer_scd2
WHERE customer_id = 7
  AND effective_date <= '2024-03-01'
  AND (end_date IS NULL OR end_date > '2024-03-01')
""")

,city,effective_date,end_date
0,Vancouver,2023-02-20,2024-06-15


## Section 4 — SCD Type 3: previous-value column

A lightweight middle ground: keep only the *immediately prior* value in a `previous_X` column,
overwriting the current one. Cheaper than Type 2 (no new rows), but only remembers one step back —
use it when you only ever need "what changed last time," not full history.

In [7]:
execute("""
CREATE TABLE dim_customer_scd3 (
    customer_key  INTEGER PRIMARY KEY,
    customer_id   INTEGER NOT NULL,
    name          VARCHAR(100) NOT NULL,
    current_city  VARCHAR(50),
    previous_city VARCHAR(50)
)
""")
execute("INSERT INTO dim_customer_scd3 VALUES (1, 7, 'Grace Kim', 'Vancouver', NULL)")

execute("""
UPDATE dim_customer_scd3
SET previous_city = current_city, current_city = 'Montreal'
WHERE customer_id = 7
""")
sql("SELECT * FROM dim_customer_scd3")

,customer_key,customer_id,name,current_city,previous_city
0,1,7,Grace Kim,Montreal,Vancouver


## Best Practices — SCD & Incremental Loading

- Default to **Type 1** unless you have a concrete need to answer "what was true historically" —
  Type 2's extra rows and query complexity aren't free.
- Type 2 needs a surrogate key (Part 2) — you cannot version a row's history if the key you join on
  *is* the thing that would need to repeat.
- Watermark loading is the cheapest incremental strategy but is blind to **hard deletes** in the
  source (a deleted row just stops appearing — it won't show up as "changed"). If deletes matter,
  you need log-based CDC.
- **Production version of everything above:** Part 10 (Simulating a Platform Locally) shows dbt's
  `incremental` materialization, which implements exactly this watermark + merge pattern as
  configuration instead of hand-written procedural logic.

## Next

**Part 9 — Query Performance & Tuning** covers indexes, partitioning, and the other side of
"why is this incremental load still slow.